In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:

from functools import partial
from typing import List, Optional
import jax
import jax.numpy as jnp
import numpy as np

from flax import nnx

from probjax.nn import GaussianFourierEmbedding, Transformer
from probjax.nn.loss_fn.denoising import build_time_dependent_denoising_loss
from probjax.nn.nets.denoising_diffusion_model import EDM
from probjax.nn import MLP
from probjax.nn.utils import AffineFuse
from dmri.simulators import MultiCompartment, AllGaussianModels


In [3]:
AllGaussianModels.fraction_prior

Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)

In [4]:
from dmri.nn.tokenizer import DMRITokenizer

In [5]:
dmri_tokenizer = DMRITokenizer(AllGaussianModels, rngs=nnx.Rngs(0))

In [6]:
num_components = len(AllGaussianModels.model_types)
num_total_components = len(AllGaussianModels.model_types) + len(AllGaussianModels.noise_types)

In [7]:
models_mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.5, shape=(100, num_total_components))

tokens_cfg = dmri_tokenizer.embed_cfgs(models_mask, jnp.ones((100, num_components)), tuple(AllGaussianModels.model_types), tuple(AllGaussianModels.noise_types), )

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


In [8]:
theta_dim = jnp.sum(AllGaussianModels.split_idx())
thetas = jax.random.normal(jax.random.PRNGKey(0), shape=(100, theta_dim))

In [9]:
tokens_params = jax.jit(dmri_tokenizer.embed_theta)(thetas, tokens_cfg)

In [10]:
tokens_params[0].shape

(19, 64)

In [11]:
dmri_tokenizer.decode_theta(tokens_params)

Array([[ 2.176,  0.433,  1.188, ..., -0.818, -0.058,  0.065],
       [-0.226, -0.311, -0.578, ...,  1.137,  0.085,  0.597],
       [-0.522, -0.075,  0.071, ...,  0.243,  0.356, -0.608],
       ...,
       [ 0.688, -2.436, -0.152, ..., -0.62 , -0.114,  0.719],
       [ 2.793,  0.339,  1.517, ..., -1.458,  0.146, -0.088],
       [ 1.442, -0.62 , -0.019, ...,  1.744, -0.185, -0.851]],      dtype=float32)

In [12]:
dmri_tokenizer.params_dims

(9, 1, 3, 3, 3, 4, 4, 4, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1)

In [51]:
from dmri.nn.dmri_reconstruction_model import DMRIInferenceModel, DMRIInferenceModelConfig

In [52]:
model = DMRIInferenceModel(DMRIInferenceModelConfig(AllGaussianModels), rngs=nnx.Rngs(0))

params = nnx.state(model, nnx.Param)

In [53]:
x = jax.random.normal(jax.random.PRNGKey(0), shape=(100,))
bvals = jax.random.uniform(jax.random.PRNGKey(0), shape=(100,), minval=0.1, maxval=1.0)
bvecs = jax.random.normal(jax.random.PRNGKey(0), shape=(100,3))
bvecs = bvecs / jnp.linalg.norm(bvecs, axis=-1, keepdims=True)

In [55]:
model(model_mask=models_mask, theta=thetas, x=x, bvals=bvals, bvecs=bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)
y (100, 1, 64)
tokens_cfg (100, 19, 64)


(Array([[-2.116, -1.159, -1.073, ..., -1.556, -0.297, -0.742],
        [ 0.429,  0.36 ,  0.684, ...,  2.094,  2.099,  2.103],
        [-0.052,  1.372,  0.417, ...,  1.978,  1.929,  1.762],
        ...,
        [ 0.634,  2.169,  1.313, ...,  1.445,  2.635,  1.503],
        [ 1.212,  1.215,  1.235, ...,  2.636,  2.932,  2.579],
        [ 0.072,  0.947,  0.575, ...,  0.51 ,  1.723,  1.104]],      dtype=float32),
 Array([[-1.356, -2.298, -1.581, ..., -0.596,  0.026, -0.081],
        [ 0.851,  0.9  ,  1.999, ...,  0.826,  0.489, -0.719],
        [ 1.771, -0.479, -0.2  , ...,  0.175,  2.049,  0.729],
        ...,
        [ 0.153,  1.991,  0.747, ..., -0.452, -0.3  , -0.862],
        [-1.029, -1.778, -2.351, ..., -1.061,  0.84 ,  0.106],
        [-0.913, -1.327,  0.436, ...,  1.267, -0.708,  1.017]],      dtype=float32))

In [56]:
model.loss_fn(params, jax.random.PRNGKey(0), models_mask, thetas, x, bvals, bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


Array([15.045, 15.43 ], dtype=float32)

In [59]:
# This is rejited
_f = jax.jit(jax.jacrev(model.loss_fn))

In [67]:

grad_per_loss = _f(params, jax.random.PRNGKey(0), models_mask, thetas, x, bvals, bvecs)

In [68]:
grad1 = jax.tree_util.tree_map(lambda x: x[0], grad_per_loss)
grad2 = jax.tree_util.tree_map(lambda x: x[1], grad_per_loss)

In [80]:
jax.tree_util.tree_reduce(lambda x, y: x + y, jax.tree_util.tree_map(lambda x: jnp.sum(jnp.abs(x)), grad1.encoder))


Array(3172.407, dtype=float32)

In [79]:
jax.tree_util.tree_reduce(
    lambda x, y: x + y,
    jax.tree_util.tree_map(lambda x: jnp.sum(jnp.abs(x)), grad2.encoder)
)

Array(1223., dtype=float32)